## 1. Connection to workspace

### Configure credential

We are using `DefaultAzureCredential` to access the workspace. 
`DefaultAzureCredential` should be capable of handling most Azure SDK authentication scenarios. 

Reference for other credentials if this does not work for you: [configure credential example](https://github.com/microsoft/promptflow/blob/main/examples/configuration.ipynb), [azure-identity reference doc](https://docs.microsoft.com/en-us/python/api/azure-identity/azure.identity?view=azure-python).

In [2]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

try:
    credential = DefaultAzureCredential()
    # Check if given credential can get token successfully.
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    # Fall back to InteractiveBrowserCredential in case DefaultAzureCredential does not work
    credential = InteractiveBrowserCredential()

### Connect to the workspace

We use a config file to connect to a workspace. The Azure ML workspace should be configured with a computer cluster. [Check this notebook for how to configure a workspace](https://github.com/microsoft/promptflow/blob/main/examples/configuration.ipynb)

In [6]:
from promptflow.azure import PFClient
import os
from dotenv import load_dotenv
from promptflow.core import AzureOpenAIModelConfiguration


load_dotenv()
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_GPT4_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT_NAME")
AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
)
# Connect to the workspace
pf = PFClient()

ValueError: credential can not be None

### Create necessary connections
A connection helps securely store and manage secret keys or other sensitive credentials required for interacting with the LLM and other external tools, for example Azure Content Safety.

In this notebook, we will use the `basic` & `eval-code-quality` flex flow, which uses the connection `open_ai_connection`.  We need to set up the connection if we haven't added it before.

To prepare your Azure OpenAI resource, follow these [instructions](https://learn.microsoft.com/en-us/azure/cognitive-services/openai/how-to/create-resource?pivots=web-portal) and get your `api_key` if you don't have one.

Go to [workspace portal](https://ml.azure.com/), click `Prompt flow` -> `Connections` -> `Create`, then follow the instruction to create your own connections. 
Learn more on [connections](https://learn.microsoft.com/en-us/azure/machine-learning/prompt-flow/concept-connections?view=azureml-api-2).

## 2. Batch run the function as a flow with multi-line data.


### Batch run with a data file (with multiple lines of test data)


In [ ]:
flow = "."  # Path to the flow directory
data = "./data.jsonl"  # Path to the data file

# Create a run with the flow and data
base_run = pf.run(
    flow=flow,
    data=data,
    column_mapping={
        "text": "${data.text}",
    },
    environment_variables={
        "AZURE_OPENAI_API_KEY": "${open_ai_connection.api_key}",
        "AZURE_OPENAI_ENDPOINT": "${open_ai_connection.api_base}",
    },
    stream=True,
)

In [ ]:
details = pf.get_details(base_run)
details.head(10)